# Chunking

In [1]:
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
from transformers import AutoTokenizer
import torch
import hashlib
import lancedb
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector
from lancedb.rerankers import ColbertReranker
import ollama
import os
import json
from tqdm.notebook import tqdm
import re, unicodedata
import subprocess


def clean_docling_chunk_strings(chunks):
    cleaned_chunks = []
    
    for chunk in chunks:
        # 2️⃣ Normalize Unicode and replace problematic punctuation
        chunk = unicodedata.normalize("NFKD", chunk).replace("\u00A0", " ")
        chunk = chunk.translate(str.maketrans({
            "–": "-", "—": "-", "‘": "'", "’": "'", "“": '"', "”": '"'
        }))

        # 3️⃣ Remove URLs (massive tokenizers killers)
        chunk = re.sub(r"http\S+", "", chunk)

        # 4️⃣ Normalize whitespace but preserve paragraphs
        chunk = re.sub(r"[ \t]+", " ", chunk)
        chunk = re.sub(r"\n\s*\n", "\n\n", chunk)  # merge single newlines, keep double
        chunk = chunk.strip()

        cleaned_chunks.append(chunk)

    return cleaned_chunks



EMBEDDING_MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
MAX_TOKENS = 2000
OLLAMA_MODEL_NAME= "chunker_full_doc"
# CHUNKS_WITH_METADATA_FILE_NAME = "preprocessed_chunks/anthropic_control_chunks_with_metadata.json"
CHUNKS_WITH_METADATA_FILE_NAME = "preprocessed_chunks/test.json"

INPUT_DIR = "input"


converter = DocumentConverter()
tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME),
    max_tokens=MAX_TOKENS # Optional, uses the max token number of the HF tokenizer by default
)
chunker = HybridChunker(
    tokenizer=tokenizer,
    merge_peers=True #Optional, defaults to true
)

study_names = [f for f in os.listdir(INPUT_DIR) if f.endswith('.pdf')]
processed_chunks=[]
try:
    with open(CHUNKS_WITH_METADATA_FILE_NAME, "r", encoding="utf-8") as f:
        processed_chunks = json.load(f)
except FileNotFoundError:
    print(f"No existing {CHUNKS_WITH_METADATA_FILE_NAME} file found, starting fresh.")
    

chunks_with_metadata = processed_chunks.copy()
processed_studies = set(chunk["document"] for chunk in processed_chunks)

study_names = [f for f in study_names if f not in processed_studies]
print(f"Found {len(processed_studies)} studies which are already processed.\nStudies which STILL need to be processed: {len(study_names)}:\n{study_names}...")


/home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.13/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_spec" in StageModelPreset has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.13/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_spec" in ObjectDetectionStagePreset has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.13/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_path" in LayoutModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config

No existing preprocessed_chunks/test.json file found, starting fresh.
Found 0 studies which are already processed.
Studies which STILL need to be processed: 25:
['A_Conceptual_Framework_and_Recommendations_for_Open_Data_and_Artifacts_in_Empirical_Software_Engineering.pdf', 'A_Feature_Fusion_Based_Indicator_for_Training-Free_Neural_Architecture_Search.pdf', 'A_Hybrid_Gaze_Distance_Estimation_via_Cross-Reference_of_Vergence_and_Depth.pdf', 'A_Resource_Allocation_Model_Based_on_Trust_Evaluation_in_Multi-Cloud_Environments.pdf', 'Electron_Paramagnetic_Resonance_Study_on_28Si_Single_Crystal_for_the_Future_Realization_of_the_Kilogram.pdf', 'Probabilistic_Artificial_Neural_Network_for_Line-Edge-Roughness-Induced_Random_Variation_in_FinFET.pdf', 'Quantitative_Evaluation_of_Line-Edge_Roughness_in_Various_FinFET_Structures_Bayesian_Neural_Network_With_Automatic_Model_Selection.pdf', 'Realization_of_a_Rubidium_Atomic_Frequency_Standard_With_Short-Term_Stability_in_10141_2_Level.pdf', 'Scalable_Re

In [ ]:
with open("tobacco_sliding_chunks_with_metadata.json", "r", encoding="utf-8") as f:
        tobacco_chunks = json.load(f)

with open("sliding_chunks_with_metadata.json", "r", encoding="utf-8") as f:
        scientific_chunks = json.load(f)

tobacco_chunks = [chunk['original_text'] for chunk in tobacco_chunks]
scientific_chunks = [chunk['original_text'] for chunk in scientific_chunks]

tobacco_chunk_token_length = [len(tokenizer.tokenizer.tokenize(chunk)) for chunk in tobacco_chunks]
scientific_chunk_token_length = [len(tokenizer.tokenizer.tokenize(chunk)) for chunk in scientific_chunks]

import numpy as np
print(f"Average chunk token count in different categories of documents\nTobacco: \t\t{np.mean(np.array(tobacco_chunk_token_length))}\nScientific papers:\t{np.mean(np.array(scientific_chunk_token_length))}")

# PREVIOUS VALUES
# Average chunk token count in different categories of documents
# Tobacco: 		420.94444444444446
# Scientific papers:	671.1837270341207

# VALUES WITH NEW ALGORITHM ON SCIENTIFIC PAPERS
# Average chunk token count in different categories of documents
# Tobacco: 		420.94444444444446
# Scientific papers:	663.5811518324607

Average chunk token count in different categories of documents
Tobacco: 		420.94444444444446
Scientific papers:	663.5811518324607


# Creating chunks and adding Metadata

As well as semantic context with ollama (Anthropic style)

In [ ]:
from codecarbon import EmissionsTracker

tracker_proposed = EmissionsTracker(
        project_name="proposed_document_slice",
        measure_power_secs=1,
        output_dir="./emissions_data"
    )


with tracker_proposed:
	for source in tqdm(study_names, desc="Chunking documents..."):        
		entire_doc = ""
		doc = converter.convert(f"{INPUT_DIR}/{source}").document
		chunks = list(chunker.chunk(dl_doc=doc))
		chunks_str = [chunk.text for chunk in chunks]
		chunks_str = clean_docling_chunk_strings(chunks_str)

		# Free up CUDA memory right after we got the results from Docling, so that Ollama can use the entire GPU
		if torch.cuda.is_available():
			torch.cuda.empty_cache()

		for chunk in tqdm(chunks, desc=f"Adding context for chunks of {source[:20]}...", leave=False):    
			entire_doc = ""
			chunk_index = chunks.index(chunk)

			context_length = 16_000 # Reduce window to save memory
			context_length = context_length - 2 * MAX_TOKENS # We need to reserve space for the chunk itself (twice, the context contains the chunk itself)
			total_context_chunk_number = context_length // (MAX_TOKENS*2) # 2x, cuz before and after the chunk

			start_index_original = chunk_index - total_context_chunk_number
			start_index_truncated = max(0, start_index_original) # Avoid index out of bounds

			end_index_original = chunk_index + total_context_chunk_number
			end_index_truncated = min(len(chunks)-1, end_index_original)

			if start_index_original < 0: # We are at the start of the document, so we need to add more chunks at the end
				end_index_truncated = min(len(chunks)-1, end_index_truncated + abs(start_index_original))
			if end_index_original > len(chunks)-1: # We are at the end of the document, so we need to add more chunks at the start
				start_index_truncated = max(0, start_index_truncated - abs(end_index_original - end_index_truncated))

			for i in range(start_index_truncated, end_index_truncated + 1):
				entire_doc += " " + chunks_str[i]

			entire_doc = "FULL DOCUMENT:\n" + entire_doc
			ollama_prompt = f"CHUNK:\n{chunks_str[chunk_index]}"
			history =  [{'role': 'user', 'content': entire_doc}, {'role': 'user', 'content': ollama_prompt}]

			response = ollama.chat(
				model=OLLAMA_MODEL_NAME,
				messages=history,
				# options={
				#     'gpu_layers': 100  # use  GPU for model layers if VRAM allows
				# }
			)
			context = response['message']['content']
			# print(f"Context for chunk: {context}")
			# ---- OWN APPROACH TO CONTEXT ----
			# text_to_embed = chunks_str[chunk_index] + "\n\n" + context # We put the context AFTER the chunk to not mess up cosine similarity but still benefit keyword search for exact matches

			# ---- ANTHROPIC'S APPROACH TO CONTEXT ----
			text_to_embed = context + "\n\n" + chunks_str[chunk_index] # The context is PREPENDED to the chunk as per Anthropic's original algporithm
			# print(context)
			pages = set(
					prov.page_no
					for doc_item in chunk.meta.doc_items
					for prov in doc_item.prov
				)
			id = hashlib.sha256(chunks_str[chunk_index].encode()).hexdigest()
			chunks_with_metadata.append({'text': text_to_embed, 'original_text':chunks_str[chunk_index], 'context':context, 'document':source, 'pages':list(pages), 'id': id})
			
		# Free up ollama from GPU memory so that Docling can semantically analyze the next doc even if it's like 100 pages
		subprocess.run(["ollama", "stop", OLLAMA_MODEL_NAME], check=True)
		
		#Total runtime: 29m 9s for 25 documents

[codecarbon WARNING @ 22:29:35] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 22:29:35] [setup] RAM Tracking...
[codecarbon INFO @ 22:29:35] [setup] CPU Tracking...
[codecarbon INFO @ 22:29:35] Tracking Intel CPU via RAPL interface
[codecarbon INFO @ 22:29:36] 	RAPL - Using 6 package domain(s) for CPU power measurement
[codecarbon INFO @ 22:29:36] 	RAPL - Selected 1 unique RAPL domain(s) after deduplication
[codecarbon INFO @ 22:29:36] 	RAPL - Monitoring domain 'package-0' (displayed as 'Processor Energy Delta_0(kWh)') via MMIO at /sys/class/powercap/intel-rapl/subsystem/intel-rapl-mmio/intel-rapl-mmio:0/energy_uj
[codecarbon INFO @ 22:29:36] [setup] GPU Tracking...
[codecarbon INFO @ 22:29:36] Tracking Nvidia GPU via pynvml
[codecarbon INFO @ 22:29:36] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Method: RAPL
                GPU Tracking Method: pynvm

Chunking documents...:   0%|          | 0/25 [00:00<?, ?it/s]

[INFO] 2026-03-05 22:29:39,803 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-05 22:29:39,806 [RapidOCR] device_config.py:57: Using GPU device with ID: 0
[INFO] 2026-03-05 22:29:39,814 [RapidOCR] download_file.py:60: File exists and is valid: /home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-05 22:29:39,815 [RapidOCR] main.py:50: Using /home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-05 22:29:40,192 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-05 22:29:40,193 [RapidOCR] device_config.py:57: Using GPU device with ID: 0
[INFO] 2026-03-05 22:29:40,194 [RapidOCR] download_file.py:60: File exists and is valid: /home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.13/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.

Adding context for chunks of A_Conceptual_Framewo...:   0%|          | 0/22 [00:00<?, ?it/s]

[codecarbon INFO @ 22:29:44] Energy consumed for RAM : 0.000028 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 22:29:44] Delta energy consumed for CPU with intel_rapl : 0.000015 kWh, power : 55.496792485015476 W
[codecarbon INFO @ 22:29:44] Energy consumed for All CPU : 0.000074 kWh
[codecarbon INFO @ 22:29:44] Energy consumed for all GPUs : 0.000048 kWh. Total GPU Power : 43.957624706246065 W
[codecarbon INFO @ 22:29:44] 0.000149 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 22:29:45] Energy consumed for RAM : 0.000033 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 22:29:45] Delta energy consumed for CPU with intel_rapl : 0.000016 kWh, power : 57.16307445518449 W
[codecarbon INFO @ 22:29:45] Energy consumed for All CPU : 0.000090 kWh
[codecarbon INFO @ 22:29:45] Energy consumed for all GPUs : 0.000064 kWh. Total GPU Power : 60.6250598413778 W
[codecarbon INFO @ 22:29:45] 0.000187 kWh of electricity and 0.000000 L of water were used since the begi

...

[codecarbon INFO @ 22:50:35] Energy consumed for RAM : 0.006929 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 22:50:35] Delta energy consumed for CPU with intel_rapl : 0.000017 kWh, power : 61.733172567073396 W
[codecarbon INFO @ 22:50:35] Energy consumed for All CPU : 0.019978 kWh
[codecarbon INFO @ 22:50:35] Energy consumed for all GPUs : 0.018873 kWh. Total GPU Power : 34.98824750643165 W
[codecarbon INFO @ 22:50:35] 0.045780 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 22:50:36] Energy consumed for RAM : 0.006935 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 22:50:36] Delta energy consumed for CPU with intel_rapl : 0.000020 kWh, power : 67.16915745502273 W
[codecarbon INFO @ 22:50:36] Energy consumed for All CPU : 0.019997 kWh
[codecarbon INFO @ 22:50:36] Energy consumed for all GPUs : 0.018883 kWh. Total GPU Power : 36.43135980623406 W
[codecarbon INFO @ 22:50:36] 0.045816 kWh of electricity and 0.000000 L of water were used since the begi

Adding context for chunks of srep05215.pdf...:   0%|          | 0/14 [00:00<?, ?it/s]

[codecarbon INFO @ 22:51:39] Energy consumed for RAM : 0.007283 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 22:51:39] Delta energy consumed for CPU with intel_rapl : 0.000017 kWh, power : 60.345741333556646 W
[codecarbon INFO @ 22:51:39] Energy consumed for All CPU : 0.021087 kWh
[codecarbon INFO @ 22:51:39] Energy consumed for all GPUs : 0.019997 kWh. Total GPU Power : 31.808885872125618 W
[codecarbon INFO @ 22:51:39] 0.048367 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 22:51:40] Energy consumed for RAM : 0.007289 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 22:51:40] Delta energy consumed for CPU with intel_rapl : 0.000017 kWh, power : 61.131434323023335 W
[codecarbon INFO @ 22:51:40] Energy consumed for All CPU : 0.021104 kWh
[codecarbon INFO @ 22:51:40] Energy consumed for all GPUs : 0.020006 kWh. Total GPU Power : 32.459166950989996 W
[codecarbon INFO @ 22:51:40] 0.048399 kWh of electricity and 0.000000 L of water were used since the b

Adding context for chunks of srep45325.pdf...:   0%|          | 0/9 [00:00<?, ?it/s]

[codecarbon INFO @ 22:52:56] Energy consumed for RAM : 0.007708 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 22:52:56] Delta energy consumed for CPU with intel_rapl : 0.000018 kWh, power : 63.301140331723346 W
[codecarbon INFO @ 22:52:56] Energy consumed for All CPU : 0.022419 kWh
[codecarbon INFO @ 22:52:56] Energy consumed for all GPUs : 0.021232 kWh. Total GPU Power : 47.348878617156 W
[codecarbon INFO @ 22:52:56] 0.051359 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 22:52:57] Energy consumed for RAM : 0.007714 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 22:52:57] Delta energy consumed for CPU with intel_rapl : 0.000018 kWh, power : 64.37482755885226 W
[codecarbon INFO @ 22:52:57] Energy consumed for All CPU : 0.022436 kWh
[codecarbon INFO @ 22:52:57] Energy consumed for all GPUs : 0.021242 kWh. Total GPU Power : 33.08227706423329 W
[codecarbon INFO @ 22:52:57] 0.051392 kWh of electricity and 0.000000 L of water were used since the beginn

In [4]:
# Save the the processed chunks in case VectorDB upload goes wrong.
# Luckily since this is a notebook, if the chunking is interrupted, we can still save the partial results here.
# Append new chunks to the existing file if it exists, otherwise create it
if os.path.exists(CHUNKS_WITH_METADATA_FILE_NAME):
    print(f"Appending to existing {CHUNKS_WITH_METADATA_FILE_NAME} file.")
    with open(CHUNKS_WITH_METADATA_FILE_NAME, "r", encoding="utf-8") as f:
        existing_data = json.load(f)
    # Avoid duplicate entries by id
    existing_ids = {chunk['id'] for chunk in existing_data}
    new_chunks = [chunk for chunk in chunks_with_metadata if chunk['id'] not in existing_ids]
    chunks_with_metadata = existing_data + new_chunks

with open(CHUNKS_WITH_METADATA_FILE_NAME, "w", encoding="utf-8") as f:
    json.dump(chunks_with_metadata, f, ensure_ascii=False, indent=2)

print(f"Results saved to {CHUNKS_WITH_METADATA_FILE_NAME}")

Results saved to sliding_chunks_with_metadata.json


# REORDER CONTEXT AND CHUNK

In [3]:
# CONVENIENCE STEP: Prepare the chunks with metadata file for Anthropic's original approach (context PREPENDED to chunk)
CHUNKS_WITH_METADATA_FILE_NAME = "preprocessed_chunks/chunks_with_metadata.json"
if os.path.exists(CHUNKS_WITH_METADATA_FILE_NAME):
    with open(CHUNKS_WITH_METADATA_FILE_NAME, "r", encoding="utf-8") as f:
        chunks_with_metadata = json.load(f)

    for chunk in chunks_with_metadata:
        chunk['text'] = chunk['context'] + "\n\n" + chunk['original_text']

    with open(f"preprocessed_chunks/anthropic_chunks_with_metadata.json", "w", encoding="utf-8") as f:
        json.dump(chunks_with_metadata, f, ensure_ascii=False, indent=2)

# Creating Database

In [2]:
from devtools import debug
registry = get_registry()
hf = registry.get("huggingface").create(name=EMBEDDING_MODEL_NAME, trust_remote_code=True, device="cuda" if torch.cuda.is_available() else "cpu")


# Define model
class MyDocument(LanceModel):
    text: str 
    vector: Vector(hf.ndims()) = hf.VectorField()
    original_text: str = hf.SourceField()
    context: str
    document: str
    id: str  # Unique identifier for the chunk


db = lancedb.connect("./db")
db.create_table("semantic_table", schema=MyDocument, mode="overwrite") # Uncomment this line when running this cell for the first time
table = db.open_table("semantic_table")

# Upload in batches with progress bar
with open(CHUNKS_WITH_METADATA_FILE_NAME, "r", encoding="utf-8") as f:
    chunks_with_metadata = json.load(f)
    debug(chunks_with_metadata[0])

batch_size = 100
for i in tqdm(range(0, len(chunks_with_metadata), batch_size), desc="Uploading chunks to VectorDB"):
    batch = chunks_with_metadata[i:i+batch_size]
    table.add(batch)

table.create_scalar_index("id", replace=True) # Index based on the chunk's id, used to manually prevent duplicates

reranker = ColbertReranker()
table.create_fts_index("text", replace=True) # Used by the reranker as well as the hybrid search's BM25 index
table.wait_for_index(["text_idx"])  # Wait for the indexing to finish

<All keys matched successfully>
[2026-02-01T23:02:54Z WARN  lance::dataset::write::insert] No existing dataset at /home/martin/projects/Quantwise/Quantwise-Chunking/db/semantic_table.lance, it will be created


/tmp/ipykernel_267651/447807103.py:23 <module>
    chunks_with_metadata[0]: {
        'text': (
            'Provides publication details and author affiliations.\n'
            '\n'
            'Received July 29, 2018, accepted August 27, 2018, date of publication September 13, 2018, date of current'
            ' version October 8, 2018.\n'
            'Digital Object Identifier 10.1 109/ACCESS.2018.2869735'
        ),
        'original_text': (
            'Received July 29, 2018, accepted August 27, 2018, date of publication September 13, 2018, date of current'
            ' version October 8, 2018.\n'
            'Digital Object Identifier 10.1 109/ACCESS.2018.2869735'
        ),
        'context': 'Provides publication details and author affiliations.',
        'document': 'Stock_Market_Prediction_via_Multi-Source_Multiple_Instance_Learning.pdf',
        'id': '990434da769b3b976cb9daaf6357417afdaad3ec0ad97db94bbba4a92877bdc2',
    } (dict) len=5


Uploading chunks to VectorDB:   0%|          | 0/4 [00:00<?, ?it/s]

<All keys matched successfully>
<All keys matched successfully>
<All keys matched successfully>
<All keys matched successfully>


Loading ColBERTRanker model colbert-ir/colbertv2.0 (this message can be suppressed by setting verbose=0)
No device set
Using device cuda
No dtype set
Using dtype torch.float32
Loading model colbert-ir/colbertv2.0, this might take a while...
Linear Dim set to: 128 for downcasting


# Example query

In [6]:
prompt = "How was stock market data gathered?"
results = table.search(prompt, query_type="hybrid", vector_column_name="vector", fts_columns="text") \
            .rerank(reranker=reranker) \
            .limit(5) \
            .to_pandas()


results

,text,vector,original_text,context,document,pages,id,_relevance_score
0,"Details the data collection process, specifica...","[0.62853956, 0.99925286, -3.499069, -0.4919791...",We collected stock market-related information ...,"Details the data collection process, specifica...",Stock_Market_Prediction_via_Multi-Source_Multi...,[6],08aa975b95d5c9c782b71fcb335e2ab61751bb4c4f6bb1...,0.990773
1,Introduces the central thesis about using Goog...,"[0.40788847, 1.8804536, -3.7192028, -0.2992431...","SUBJECT AREAS:\nSTATISTICAL PHYSICS, THERMODYN...",Introduces the central thesis about using Goog...,srep01684.pdf,[1],326e42cc95fc78ae06dc4023c715a91f02054804d2d494...,0.989308
2,Suggests that Google Trends data may provide i...,"[0.18005954, 2.2202637, -3.539316, -0.22429425...","In summary, our results are consistent with th...",Suggests that Google Trends data may provide i...,srep01684.pdf,[5],c80c4fb4f9449b543440f05257d57a5bee14a10d6f666e...,0.944270
3,"This section details the experimental design, ...","[0.58569866, 1.1435417, -2.793894, -0.5842912,...",Experimental design. Our paper relates to rese...,"This section details the experimental design, ...",s41598-020-77823-3.pdf,"[4, 5]",a3a29e665eb3215584b2cfab75bcaa99692dd7f3757763...,0.897300
4,Quantifies trading behavior in financial marke...,"[0.5763039, 1.8760982, -3.197494, -0.5494522, ...",We analyze the performance of a set of 98 sear...,Quantifies trading behavior in financial marke...,srep01684.pdf,"[1, 2]",293d693229ed7fd9ac926515d24b56a9425b831c435fb9...,0.894645


In [8]:
results.iloc[0,0]

'We collected stock market-related information from Jan. 1, 2015 to Dec. 31, 2016, and separate the information into two data sets, one for the year 2015 and the other for 2016. The data consist of three parts, the historical quantitative data, the news articles and the posts on the social network, which are introduced in detail as follows.\n- GLYPH<15> Quantitative data : the source of quantitative data is Wind, 2 a widely used GLYPH<28>nancial information service provider in China. The data we collect are the average prices, market index change and turnover rate of the Shanghai Composite Index in each trading day.\n- GLYPH<15> News data : we collect the news articles on the macro economy through Wind, and get 38,727 and 39,465 news articles in 2015 and 2016 respectively. The news articles are aggregated by Wind from major GLYPH<28>nancial news websites in China, such as and We process the news titles rather than the whole articles to extract the events, as the main topic of a news ar

In [9]:
table.stats()

{'total_bytes': 3504703,
 'num_rows': 382,
 'num_indices': 2,
 'fragment_stats': {'num_fragments': 4,
  'num_small_fragments': 4,
  'lengths': {'min': 82,
   'max': 100,
   'mean': 95,
   'p25': 100,
   'p50': 100,
   'p75': 100,
   'p99': 100}}}